In [2]:
# somthine a single query might not capture all the ways info is phrased in your documents it reduse the ambiguirty 
# generate multiple query after taking the query from the user we have multiple query now it search for the same document as all the retrivers / 


In [5]:
!pip install -U langchain


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_classic.retrievers import MultiQueryRetriever

In [7]:
import langchain
print(langchain.__version__)

1.3.14


In [8]:
all_docs = [
    Document(
        page_content="Regular walking boosts heart health and can reduce symptoms of depression.",
        metadata={"source": "H1"},
    ),
    Document(
        page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.",
        metadata={"source": "H2"},
    ),
    Document(
        page_content="Deep sleep is crucial for cellular repair and emotional regulation.",
        metadata={"source": "H3"},
    ),
    Document(
        page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.",
        metadata={"source": "H4"},
    ),
    Document(
        page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.",
        metadata={"source": "H5"},
    ),
    Document(
        page_content="The solar energy system in modern homes helps balance electricity demand.",
        metadata={"source": "I1"},
    ),
    Document(
        page_content="Python balances readability with power, making it a popular system design language.",
        metadata={"source": "I2"},
    ),
    Document(
        page_content="Photosynthesis enables plants to produce energy by converting sunlight.",
        metadata={"source": "I3"},
    ),
    Document(
        page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.",
        metadata={"source": "I4"},
    ),
    Document(
        page_content="Black holes bend spacetime and store immense gravitational energy.",
        metadata={"source": "I5"},
    ),
]

In [9]:
embidding_model=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore=FAISS.from_documents(
        documents=all_docs,
        embedding=embidding_model,    
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3098.54it/s]


In [14]:
similarity_retriever=vectorstore.as_retriever(search_type="similarity",search_kwargs={'k':5})

In [10]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

pipe = pipeline(
    "text-generation",
    model="sentence-transformers/all-MiniLM-L6-v2",
)

llm = HuggingFacePipeline(pipeline=pipe)

[transformers] If you want to use `BertLMHeadModel` as a standalone, add `is_decoder=True.`
Loading weights: 100%|██████████| 101/101 [00:00<00:00, 1541.07it/s]
[transformers] This checkpoint seem corrupted. The tied weights mapping for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are absent from the checkpoint, and we could not find another related tied weight for those keys
[transformers] BertLMHeadModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                                        | Status     | 
-------------------------------------------+------------+-
pooler.dense.bias                          | UNEXPECTED | 
pooler.dense.weight                        | UNEXPECTED | 
cls.predictions.transform.dense.bias       | MISSING    | 
cls.predictions.transform.LayerNorm.bias   | MISSING    | 
cls.predictions.transform.dense.weight     | MISSING    | 
cls.predictions.transform.LayerNorm.weight | MISSING    | 
cls.predictions

In [11]:
from langchain_classic.retrievers import MultiQueryRetriever
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={'k':5}),
    llm=llm
    )


In [ ]:
query="How to improve energy levels and maintain  balance?" #ambiguos

In [15]:
similarity_results=similarity_retriever.invoke(query)
multiquery_result=multiquery_retriever.invoke(query)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [16]:
for i, doc in enumerate(multiquery_result):
    print(f"n--- results {i+1}")
    print(doc.page_content)

n--- results 1
Python balances readability with power, making it a popular system design language.
n--- results 2
Mindfulness and controlled breathing lower cortisol and improve mental clarity.
n--- results 3
The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.
n--- results 4
Drinking sufficient water throughout the day helps maintain metabolism and energy.
n--- results 5
Photosynthesis enables plants to produce energy by converting sunlight.
n--- results 6
Consuming leafy greens and fruits helps detox the body and improve longevity.
n--- results 7
Black holes bend spacetime and store immense gravitational energy.
n--- results 8
The solar energy system in modern homes helps balance electricity demand.
n--- results 9
Regular walking boosts heart health and can reduce symptoms of depression.


In [ ]:
for i, doc in enumerate(similarity_results):
    print(f"n--- results {i+1}")
    print(doc.page_content) #get confused 

n--- results 1
Drinking sufficient water throughout the day helps maintain metabolism and energy.
n--- results 2
The solar energy system in modern homes helps balance electricity demand.
n--- results 3
Consuming leafy greens and fruits helps detox the body and improve longevity.
n--- results 4
Mindfulness and controlled breathing lower cortisol and improve mental clarity.
n--- results 5
Photosynthesis enables plants to produce energy by converting sunlight.
